# The Architects single task Test-time Training

I simply took the solution for ARC24 from the Architects and made
a few small modifications:

- Apply test-time training to each task individually instead of training for a group of tasks together.
- Modify it to work efficiently on 4 GPUs.
- Hyperparameter tuning.


More info here: https://ironbar.github.io/arc25/05_Solution_Summary/

In [ ]:
!cp /kaggle/input/the-architects/*.py .

In [ ]:
import os
import random
random_seed = random.randint(0, 2**32-1)
os.environ['RANDOM_SEED'] = str(random_seed)
print(f'Random seed {random_seed} will be used in this run')

In [ ]:
%%writefile common_stuff.py
# common configuration for training and evaluation
import time
import fcntl
from arc_loader import *
from model_runner import *
from selection import *
from async_tools import *
from easy_logging import logging, log_execution_time
logger = logging.getLogger(__name__)

# paths
tmp_dir = '/kaggle/temp'
arc_challenge_file = '/kaggle/input/arc-prize-2025/arc-agi_test_challenges.json'
model_temp_storage = os.path.join(tmp_dir, 'finetuned_model')
infer_temp_storage = os.path.join(tmp_dir, 'inference_outputs')
score_temp_storage = os.path.join(tmp_dir, 'inference_scoring')
train_tasks_lock_storage = os.path.join(tmp_dir, 'train_locks')
infer_tasks_lock_storage = os.path.join(tmp_dir, 'infer_locks')

# models
#base_model = '/kaggle/input/wb55l_nemomini_fulleval/transformers/default/1'
base_model = '/dev/shm/wb55l_nemomini_fulleval/1'
MyFormatter, perm_aug = ArcFormatter_premix_3, 'rnd_all'
mask_first = 0

# training & inference
use_evaluation_set = False
train_epochs = 10 # default 4
n_gpus = 4
train_gpu_slots = 2
infer_gpu_slots = 3
max_seq_length_infer = 8192 # default 8192
max_seq_length_train = 8192 # default 4224
infer_params = dict(min_prob=0.13, # default 0.17
                    store=infer_temp_storage, use_turbo=True)

random_seed = int(os.environ['RANDOM_SEED'])
logger.info(f'Using random seed: {random_seed}')
infer_aug_params = dict(n=1, tp=True, rot=True, seed=random_seed+1, perm=perm_aug, shfl_ex=True)
lora_rank = 4
learning_rate = 2e-4 # default 1e-4


# scoring
use_aug_score = True
aug_score_params = dict(tp=True, rot=True, perm=perm_aug, shfl_ex=True, make_unique=True, max_len=max_seq_length_infer)
submission_select_algo = score_full_probmul_3 if use_aug_score else score_all_probsum



if use_evaluation_set:
    os.environ['KAGGLE_IS_COMPETITION_RERUN'] = 'True'

def load_dataset():
    if use_evaluation_set:
        arc_test_set = ArcDataset.from_file('/kaggle/input/arc-prize-2025/arc-agi_evaluation_challenges.json')
        arc_test_set.load_replies('/kaggle/input/arc-prize-2025/arc-agi_evaluation_solutions.json')
        arc_test_set = arc_test_set.sorted_by_len(formatter=None, name='input', max_of_transposed=True)
        already_solved_tasks = {'800d221b', '16b78196', '6e453dd6', '7666fa5d', '38007db0', '53fb4810', 
                                '8f3a5a89', '78332cb0', '0934a4d8', '7ed72f31', '136b0064', '981571dc', 
                                'd35bdbdc', 'db0c5428', '2d0172a1', '3a25b0d8', '1818057f', 'aa4ec2a5', 
                                '8b9c3697', '71e489b6', '1ae2feb7', '36a08778'}
        arc_test_set = arc_test_set.change_keys(already_solved_tasks)
        #print('arc_test_set.get_lengths', arc_test_set.get_lengths(formatter=None, name='input', max_of_transposed=True))
        arc_test_set.is_orig = True
    else:
        arc_test_set = ArcDataset.from_file(arc_challenge_file)
    return arc_test_set


def prepare_run(model_path, load_lora=None, train=False, gpu=None, **kwargs):
    if gpu is not None:
        os.environ["CUDA_DEVICE_ORDER"   ] = "PCI_BUS_ID"
        os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)

    model, tokenizer, formatter = prepare_model(  # base model configuration
        model=model_path,
        local_files_only=True,
        mode='unsloth_4bit', #default unsloth_4bit
        #shrink_embedding=8000,
        max_seq_length=max_seq_length_train,
        formatter=MyFormatter,
        peft=([dict(
            r=lora_rank,  # default 64
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 
                            'gate_proj', 'up_proj', 'down_proj', 'embed_tokens', 'lm_head'],
            lora_alpha=16,
            lora_dropout=0,  # Supports any, but = 0 is optimized
            bias="none",  # Supports any, but = "none" is optimized
            use_gradient_checkpointing=True,  # default=True or "unsloth" for very long context
            random_state=42,
            use_rslora=True,  # We support rank stabilized LoRA
            loftq_config=None,  # And LoftQ
        )] if train or load_lora else []) + ([load_lora] if load_lora else []),
        **kwargs
    )
    
    if train and mask_first: formatter.collator_kwargs.update(mask_first_output=mask_first)
    return model, formatter


def print_dataset_parameters(arc_test_set):
    parameters = {key: get_task_properties(arc_test_set.queries[key]) for key in arc_test_set.keys}
    print('Dataset parameters', parameters)


def get_task_properties(task):
    output_input_ratio = np.mean([np.prod(np.shape(sample['output']))/np.prod(np.shape(sample['input'])) for sample in task['train']])
    properties = {
        'train': sum(np.prod(np.shape(v)) for v2 in task['train'] for v in v2.values()),
        'n_test': len(task['test']),
        'total': sum(np.prod(np.shape(v)) for v3 in task.values() for v2 in v3 for v in v2.values()),
        'test_output': int(output_input_ratio*sum(np.prod(np.shape(v)) for v2 in task['test'] for v in v2.values()))
    }
    return properties


def prepare_dataset(formatter, train, key, seed=random_seed):
    arc_test_set = load_dataset()
    ds = arc_test_set
    ds = ds.change_keys([key])
    if train:
        ds = ds.remove_replies()
        logger.info(f'Keys before data augmentation={seed}: {ds.keys}')
        ds = ds.augment(tp=True, rot=True, perm=perm_aug, n=train_epochs, shfl_ex=True, shfl_keys=True, seed=random_seed)
        ds = ds.cut_to_len(formatter=formatter, name='text', max_len=max_seq_length_train, max_new_tokens=0)
        logger.info(f'Keys that will be used for training={seed}: {ds.keys}')
    else:
        ds = ds.split_multi_replies()
        #logger.info(f'ds.length() before augmentation: {ds.length()}')
        ds = ds.augment(**infer_aug_params).interleave(ds.length())
        #logger.info(f'ds.length() after augmentation: {ds.length()}')
        #logger.info(f'Keys after augment with seed={seed}: {ds.keys}')
        ds = ds.cut_to_len(formatter=formatter, name='input', max_len=max_seq_length_infer)
    return ds, arc_test_set


def acquire_gpu(slots_per_gpu):
    while True:
        for slot in range(slots_per_gpu):
            for gpu in range(n_gpus):
                try:
                    lock_file = f"/tmp/gpu{gpu}_slot{slot}_lock"
                    f = open(lock_file, 'w')
                    fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
                    logger.info(f"Acquired GPU{gpu} slot{slot}")
                    return gpu, slot, f  # Return both GPU ID and lock file handle
                except BlockingIOError:
                    time.sleep(0.1) 
                    continue  # Lock not available, try next GPU
        time.sleep(1) # No GPU available, sleep and retry


def create_locks(ds):
    os.makedirs(infer_tasks_lock_storage)
    os.makedirs(train_tasks_lock_storage)
    ds = ds.sorted_by_len(formatter=None, name='input', max_of_transposed=True, reverse=True)
    for idx, key in enumerate(ds.keys):
        open(os.path.join(infer_tasks_lock_storage, f'{idx:03d}_{key}.lock'), "w").close()
        open(os.path.join(train_tasks_lock_storage, f'{idx:03d}_{key}.lock'), "w").close()


def find_and_lock_next_task(tasks_dir, reverse=False):
    task_paths = sorted(glob(os.path.join(tasks_dir, '*.lock')))
    if reverse: task_paths = task_paths[::-1]
    
    for task_path in task_paths:
        if os.path.exists(task_path + '.done'):
            continue
        try:
            f = open(task_path, "r+")
            # Attempt to acquire non-blocking exclusive lock
            fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
            task_id = os.path.splitext(os.path.basename(task_path))[0].split('_')[-1]
            # If we reach here, we have the lock
            return task_id, task_path, f
        except BlockingIOError:
            # Another process has locked it, skip
            continue

    # No task available
    return None, None, None

    
@log_execution_time
def start_training():
    try:
        gpu, slot, lock_file = acquire_gpu(slots_per_gpu=train_gpu_slots)
        try:
            task_id, task_path, task_lock = find_and_lock_next_task(train_tasks_lock_storage, reverse=slot==1)
            logger.info(f'Going to train task {task_id} on gpu{gpu} slot{slot}')
            try:
                storage_path = get_model_storage_path(task_id)
                if not os.path.exists(storage_path):
                    with RemapCudaOOM():
                        model, formatter = prepare_run(base_model, train=True, gpu=gpu)
                        dataset, _ = prepare_dataset(formatter, train=True, key=task_id)
                        model, trainer_stats = training_run(
                            model, formatter, dataset, store=storage_path,
                            max_seq_length=max_seq_length_train,
                            grad_acc_fix=False,
                            train_args=dict(
                                per_device_train_batch_size=1, # default=2
                                gradient_accumulation_steps=1, # default=2
                                #warmup_steps=100, # On ARC24 they trained for 400 step (50*8*4/4)
                                warmup_ratio=0.1,
                                num_train_epochs=1,
                                max_steps=-1,
                                learning_rate=learning_rate, # default 1e-4
                                embedding_learning_rate=learning_rate/10, # default 1e-5
                                logging_steps=10,
                                optim="adamw_8bit",
                                weight_decay=0.01,  # 0.01,
                                lr_scheduler_type='cosine',  # "linear", "cosine",
                                seed=random_seed,
                                output_dir=os.path.join(tmp_dir, 'checkpoints'),
                                save_strategy="no",
                                report_to='none',
                            ),
                        )
                        mem_info()
                with open(task_path + '.done', 'w') as f:
                    f.write('done')
            finally:
                task_lock.close()
        finally:
            logger.info(f"Releasing GPU {gpu}")
            lock_file.close()
    finally: os.makedirs(f'{storage_path}_done', exist_ok=True)


def check_gpu_finished_training(gpu, sleep=1):
    n_tasks = len(load_dataset().keys)
    while True:
        finished_filepaths = glob(f'{model_temp_storage}_*_gpu{gpu}_done')
        if len(finished_filepaths) >= n_tasks//n_gpus:
            logger.info(f'GPU {gpu} training was completed ({finished_filepaths})')
            break
        time.sleep(sleep)


def get_model_storage_path(key):
    return f'{model_temp_storage}_{key}'

    
@log_execution_time
def start_inference(seed=random_seed):
    gpu, slot, lock_file = acquire_gpu(slots_per_gpu=infer_gpu_slots)
    try:
        task_id, task_path, task_lock = find_and_lock_next_task(infer_tasks_lock_storage, reverse=slot==1)
        logger.info(f'Going to do inference for task {task_id} on gpu{gpu} slot{slot}')
        try:
            storage_path = get_model_storage_path(task_id)
            with RemapCudaOOM():
                model, formatter = prepare_run(storage_path, gpu=gpu)
                dataset, arc_test_set = prepare_dataset(formatter, train=False, key=task_id, seed=seed)
                retrainer = None # I have removed a lot of code from here
                decoder = Decoder(formatter, arc_test_set.split_multi_replies(), n_guesses=2, prob_baseline=0.05)
                inference_run_v2(model, formatter, dataset, decoder, retrain=retrainer, **infer_params)
                if use_aug_score: decoder.calc_augmented_scores(model=model, store=score_temp_storage, **aug_score_params)
                mem_info()
            with open(task_path + '.done', 'w') as f:
                f.write('done')
        finally:
            task_lock.close()
    finally:
        logger.info(f"Releasing GPU {gpu}")
        lock_file.close()


class RemapCudaOOM:
    def __enter__(self): pass
    def __exit__(self, exc_type, exc_value, traceback):
        oom_errors = ["CUDA out of memory", "Make sure you have enough GPU RAM", "does not fit any GPU's remaining memory"]
        if exc_value and any(x in str(exc_value) for x in oom_errors):
            with open('submission.json', 'w') as f: f.write('cause submission scoring error')

In [ ]:
%%time
import os
import subprocess
from common_stuff import *

os.environ["WANDB_DISABLED"] = "true"

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    if not os.path.exists(os.path.join(tmp_dir, 'unsloth_installed')):  # unsloth offline install - https://stackoverflow.com/a/51646354
        !pip uninstall --yes torch accelerate
        !pip install --no-index --find-links=/kaggle/input/unsloth-2024-9-post4/wheelhouse unsloth
        #!pip uninstall --yes accelerate fastai torch torchaudio transformers
        #!pip install --no-index --find-links=/kaggle/input/unsloth-2024-10-7/wheelhouse unsloth  # do not use grad_acc_fix - trains very slow
        #!sed -i 's/if ((post_check - pre_check) >= 1).sum() > 1:/if False:/g' /opt/conda/lib/python3.10/site-packages/unsloth/models/llama.py
        # fix delay bug in get_statistics()
        !sed -i 's/^def get_statistics():/def get_statistics():\n if False:/g' /opt/conda/lib/python3.10/site-packages/unsloth/models/_utils.py
        # fix faulty unsloth multi-gpu detection
        !sed -i "s/raise RuntimeError('Unsloth currently does not support multi GPU setups - but we are working on it!')/pass/g" /opt/conda/lib/python3.10/site-packages/unsloth/tokenizer_utils.py /opt/conda/lib/python3.10/site-packages/unsloth/models/llama.py /opt/conda/lib/python3.10/site-packages/unsloth/models/vision.py
        os.makedirs(os.path.join(tmp_dir, 'unsloth_installed'), exist_ok=True)
        print('Unsloth installed & patched.')
        !pip install -q --no-index --find-links=/kaggle/input/making-wheels-of-necessary-packages-for-vllm GPUtil

        !mkdir /dev/shm/wb55l_nemomini_fulleval
        !cp -r '/kaggle/input/wb55l_nemomini_fulleval/transformers/default/1' /dev/shm/wb55l_nemomini_fulleval

In [ ]:
def get_gpu_count():
    result = subprocess.run(["nvidia-smi", "-L"], stdout=subprocess.PIPE, text=True)
    gpu_list = result.stdout.strip().split('\n')
    print(result.stdout)
    return len(gpu_list)

# Avoid failing submissions by checking the number of gpus
assert n_gpus == get_gpu_count()

In [ ]:
# Train
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    from resource_monitor import ResourceMonitor
    
    monitor = ResourceMonitor(interval=1)
    monitor.start()
    
    arc_test_set = load_dataset()
    #print_dataset_parameters(arc_test_set)
    
    !rm -r {model_temp_storage}* {infer_tasks_lock_storage} {train_tasks_lock_storage}
    create_locks(arc_test_set)
    # Launch a training process and an inference process for each GPU.
    t0 = time.time()
    train_processes, infer_processes = [], []
    for idx, _ in enumerate(arc_test_set.keys):
        train_cmd = f"from common_stuff import start_training; start_training()"
        train_proc = await spawn_process(train_cmd, f"train_{idx}")
        train_processes.append(train_proc)
    proc_exit_codes = await wait_for_subprocesses(*train_processes
                                                  , print_output=True)
    assert all(code == 0 for code in proc_exit_codes), "One or more subprocesses failed!"
    print(f'It took {int(time.time() - t0)} seconds to run training with {n_gpus} GPUs')

In [ ]:
# Inference
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !rm -r {infer_temp_storage} {score_temp_storage}*
    t0 = time.time()
    for idx, _ in enumerate(arc_test_set.keys):
        infer_cmd = f"from common_stuff import start_inference; start_inference()"
        infer_proc = await spawn_process(infer_cmd, f"infer_{idx}")
        infer_processes.append(infer_proc)
    proc_exit_codes = await wait_for_subprocesses(*infer_processes, print_output=True)
    print(f'*** Subprocesses exit codes: {proc_exit_codes}')
    
    # Optionally, assert all exit codes are 0.
    assert all(code == 0 for code in proc_exit_codes), "One or more subprocesses failed!"
    print(f'It took {int(time.time() - t0)} seconds to run inference with {n_gpus} GPUs')

In [ ]:
!tree -r -h /kaggle/temp

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print(f'Number of predictions: {len(glob(os.path.join(infer_temp_storage, "*")))}')
    print(f'Number of valid outputs: {len(glob(os.path.join(f"{score_temp_storage}_new", "*")))}')

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # write submission
    from common_stuff import *
    with RemapCudaOOM():
        model, formatter, dataset = None, MyFormatter(), None
        decoder = Decoder(formatter, arc_test_set.split_multi_replies(),
                          n_guesses=2, frac_score=True).from_store(infer_params['store'])
        if use_aug_score or arc_test_set.is_fake: decoder.calc_augmented_scores(
            model=model, store=score_temp_storage, **aug_score_params)
        submission = arc_test_set.get_submission(decoder.run_selection_algo(submission_select_algo))
        with open('submission.json', 'w') as f: json.dump(submission, f)
        if arc_test_set.is_fake:
            decoder.benchmark_selection_algos(selection_algorithms)
            with open('submission.json') as f: reload_submission = json.load(f)
            print('*** Reload score:', arc_test_set.validate_submission(reload_submission))
else:
    import json
    with open('submission.json', 'w') as f:
        json.dump(dict(dry_run=True), f)

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN') and use_evaluation_set:
    def load_json(filepath):
        with open(filepath, 'r') as f:
            return json.load(f)
    
    def evaluate(ground_truth, submission):
        comparison = dict()
        for key, predictions in submission.items():
            comparison[key] = []
            solutions = ground_truth[key]
            for idx, solution in enumerate(solutions):
                comparison[key].append(any(prediction == solution for prediction in predictions[idx].values()))
        task_scores = {key: np.mean(values) for key, values in comparison.items()}
        print(f'Mean score: {np.mean(list(task_scores.values())):.1%}')
        task_above_zero = {key: values for key, values in task_scores.items() if np.mean(values) > 0}
        print(f'Tasks with non-zero score {len(task_above_zero)}: {task_above_zero}')
        return task_scores

    submission = load_json('submission.json')
    print(f'Number of tasks: {len(submission)}')
    ground_truth = load_json('/kaggle/input/arc-prize-2025/arc-agi_evaluation_solutions.json')
    evaluate(ground_truth, submission);

In [ ]:
!rm -r *.py __pycache__ __notebook__.ipynb
!ls

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    monitor.stop()
    monitor.plot()
    #monitor.save()